# Week 8 - Task 2: Base Model Exploration

This notebook is meant to run in Google Colab with a T4 GPU. It covers Task 2.1 (multi-domain probing) and Task 2.2 (GSM8K zero-shot baseline).

## Colab Setup

- Use a T4 runtime before loading the model.
- Upload `inference.py` and `evaluation.py` from this repo before running the import cell.
- The probe prompts are defined inside this notebook, so you do not need `prompts.py` in Colab.

In [1]:
# Check the current Unsloth README before running if this install cell stops working.
!pip install unsloth -q
!pip install groq datasets evaluate rouge_score nltk -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [2]:
SEED = 42
TRAINING_SEED = 3407

import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel

np.random.seed(SEED)
_ = torch.manual_seed(SEED)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

assert torch.cuda.is_available(), "No GPU found. Switch the Colab runtime to T4 before running this notebook."
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU:  Tesla T4
VRAM: 15.6 GB


In [3]:
repo_root = Path.cwd().resolve()
possible_src = [repo_root / "src", repo_root.parent / "src", repo_root]
for candidate in possible_src:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.append(str(candidate))

from evaluation import evaluate_gsm8k_sample
from inference import generate_response, run_probe_set


## Load the Base Model

The default model below matches the recommended T4 setup from the deliverables. Change `MODEL_NAME` only if you have a clear reason and document that choice.

In [4]:
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(model)

print({
    "model_name": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "load_in_4bit": True,
})

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
{'model_name': 'unsloth/Qwen2.5-1.5B-Instruct', 'max_seq_length': 2048, 'load_in_4bit': True}


## Task 2.1 - Multi-Domain Failure Inventory

In [5]:
TASK2_GENERAL_SYSTEM_PROMPT = (
    "You are a helpful assistant. Follow the user's instructions carefully."
)

GSM8K_ZERO_SHOT_SYSTEM_PROMPT = (
    "Solve the following math problem step by step. "
    "Show your reasoning, then write your final answer after '####'."
)

TASK2_PROBES = {
    "math_reasoning": [
        {
            "id": "math_1",
            "prompt": (
                "A bakery makes 96 muffins. It packs them equally into 12 boxes. "
                "Then 3 muffins are removed from each box before delivery. "
                "How many muffins are delivered in total?"
            ),
        },
        {
            "id": "math_2",
            "prompt": (
                "A train travels 180 kilometers in 3 hours. At the same speed, "
                "how far will it travel in 7.5 hours? Show the steps."
            ),
        },
        {
            "id": "math_3",
            "prompt": (
                "Solve this and end with the format '#### answer': "
                "Lina had 45 stickers, gave 18 away, and then bought 27 more. "
                "How many stickers does she have now?"
            ),
        },
        {
            "id": "math_4",
            "prompt": (
                "A class has 14 boys and 18 girls. One quarter of the students are absent. "
                "How many students are present? Explain each step."
            ),
        },
    ],
    "code": [
        {
            "id": "code_1",
            "prompt": (
                "Write a Python function `is_palindrome(text: str) -> bool` that ignores "
                "spaces, punctuation, and case. Include two small tests."
            ),
        },
        {
            "id": "code_2",
            "prompt": (
                "Write a Python function that returns the second largest unique number in "
                "a list of integers. If it does not exist, raise `ValueError`."
            ),
        },
        {
            "id": "code_3",
            "prompt": (
                "Translate this Python function into idiomatic C#:\n\n"
                "def running_sum(values):\n"
                "    total = 0\n"
                "    out = []\n"
                "    for value in values:\n"
                "        total += value\n"
                "        out.append(total)\n"
                "    return out"
            ),
        },
    ],
    "instruction_following": [
        {
            "id": "instruction_1",
            "prompt": (
                "Answer with exactly 3 bullet points. Each bullet must be at most 6 words. "
                "Topic: why version control is useful."
            ),
        },
        {
            "id": "instruction_2",
            "prompt": (
                "Return only valid JSON with keys `title` and `year`. "
                "Set `title` to `Dune` and `year` to 1965."
            ),
        },
        {
            "id": "instruction_3",
            "prompt": (
                "Write one sentence about the ocean. Do not use the letter e."
            ),
        },
    ],
    "surprising_competence": [
        {
            "id": "competence_1",
            "prompt": (
                "Explain in simple terms what a SQL LEFT JOIN does, and include one tiny example."
            ),
        },
        {
            "id": "competence_2",
            "prompt": (
                "Write a regular expression that matches a date in the format YYYY-MM-DD, "
                "and briefly explain each part."
            ),
        },
        {
            "id": "competence_3",
            "prompt": (
                "Summarize the difference between stacks and queues in 4 short sentences."
            ),
        },
    ],
}

probe_counts = {domain: len(items) for domain, items in TASK2_PROBES.items()}
probe_counts


{'math_reasoning': 4,
 'code': 3,
 'instruction_following': 3,
 'surprising_competence': 3}

In [6]:
math_results = run_probe_set(
    model=model,
    tokenizer=tokenizer,
    probes=TASK2_PROBES["math_reasoning"],
    system_prompt=TASK2_GENERAL_SYSTEM_PROMPT,
    max_new_tokens=500,
)

for result in math_results:
    print("=" * 100)
    print(f"ID: {result['id']}")
    print("PROMPT:")
    print(result["prompt"])
    print()
    print("MODEL OUTPUT:")
    print(result["output"])
    print()

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

ID: math_1
PROMPT:
A bakery makes 96 muffins. It packs them equally into 12 boxes. Then 3 muffins are removed from each box before delivery. How many muffins are delivered in total?

MODEL OUTPUT:
To determine how many muffins are delivered, we need to follow these steps:

1. **Calculate the number of muffins per box:**
   The bakery makes 96 muffins and packs them equally into 12 boxes.
   \[
   \text{Muffins per box} = \frac{96}{12} = 8
   \]

2. **Determine the number of muffins left after removing 3 from each box:**
   Each box originally has 8 muffins, but 3 are removed from each box.
   \[
   \text{Remaining muffins per box} = 8 - 3 = 5
   \]

3. **Calculate the total number of muffins delivered:**
   There are 12 boxes, and each box now contains 5 muffins.
   \[
   \text{Total muffins delivered} = 12 \times 5 = 60
   \]

Thus, the total number of muffins delivered is:
\[
\boxed{60}
\]

ID: math_2
PROMPT:
A train travels 180 kilometers in 3 hours. At the same speed, how far will 

### Math Observations

#### `math_1`
- Failure type: correct
- Diagnosis: The model solved this cleanly and kept the steps consistent from start to finish. This suggests simple multi-step arithmetic with small integers is already handled well by the base model.
- Training data that could help: More examples are not urgently needed for problems at this difficulty, but similar worked examples with explicit final-answer formatting could make the response style more consistent.

#### `math_2`
- Failure type: correct
- Diagnosis: The model correctly converted distance and time into a rate and then reused that rate in the second step. This is a good sign for simple proportional reasoning when the problem structure is very direct.
- Training data that could help: Harder rate problems with distractors, extra numbers, or less direct wording would be more useful than more easy cases like this one.

#### `math_3`
- Failure type: formatting
- Diagnosis: The arithmetic is correct, but the model ignored the requested `#### answer` ending. This means the problem is not mathematical here; it is instruction-following and output-format control.
- Training data that could help: Supervised examples that always end with `#### final_number` would likely help the model learn the required GSM8K-style answer format.

#### `math_4`
- Failure type: correct
- Diagnosis: The model correctly summed the class size, computed one quarter as absent, and subtracted to get the final answer. This shows decent performance on short word problems with one clean fraction step.
- Training data that could help: More nested word problems with multiple hidden steps would test and improve deeper reasoning better than more easy fraction examples.


In [7]:
code_results = run_probe_set(
    model=model,
    tokenizer=tokenizer,
    probes=TASK2_PROBES["code"],
    system_prompt=TASK2_GENERAL_SYSTEM_PROMPT,
    max_new_tokens=500,
)

for result in code_results:
    print("=" * 100)
    print(f"ID: {result['id']}")
    print("PROMPT:")
    print(result["prompt"])
    print()
    print("MODEL OUTPUT:")
    print(result["output"])
    print()

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ID: code_1
PROMPT:
Write a Python function `is_palindrome(text: str) -> bool` that ignores spaces, punctuation, and case. Include two small tests.

MODEL OUTPUT:
```python
import re

def is_palindrome(text: str) -> bool:
    """
    Checks if the given text is a palindrome, ignoring spaces, punctuation, and case.
    
    Parameters:
    text (str): The string to check for being a palindrome.
    
    Returns:
    bool: True if the text is a palindrome, False otherwise.
    """
    # Remove non-alphanumeric characters and convert to lowercase
    cleaned_text = re.sub(r'[^a-zA-Z0-9]', '', text).lower()
    
    # Check if the cleaned text is equal to its reverse
    return cleaned_text == cleaned_text[::-1]

# Test cases
print(is_palindrome("A man, a plan, a canal: Panama"))  # Should be True
print(is_palindrome("No lemon, no melon"))             # Should be True
print(is_palindrome("Hello, world!"))                  # Should be False
```

In this code snippet, we define a function `is

### Code Observations

#### `code_1`
- Diagnosis: The function itself is correct, but the explanation after the code contradicts the actual test cases. In particular, it wrongly claims `"No lemon, no melon"` should return False even though the function would return True.
- What is broken or missing: The main issue is explanation quality, not code generation. The model produced working code, but its natural-language explanation was careless and internally inconsistent.
- Training data that could help: Paired examples where code and explanation must stay aligned would likely help reduce this kind of confident mismatch.

#### `code_2`
- Diagnosis: The function passes the included tests, but the logic is not really selecting the second largest unique value. It sorts `(value, index)` pairs, so repeated largest values can still break the intended behavior on other inputs.
- What is broken or missing: The model solved the visible tests, but the implementation is brittle because it does not truly remove duplicates before choosing the second largest unique number.
- Training data that could help: More examples with edge cases and hidden tests around duplicates would help the model learn robust list-processing logic instead of test-fitting shortcuts.

#### `code_3`
- Diagnosis: The translation is mostly correct and readable, but it is more verbose than necessary and not fully idiomatic because it adds a full `Program` wrapper and uses `long` without any clear need.
- What is broken or missing: Nothing major is broken, but the answer leans toward a safe template rather than the simplest clean translation of the original function.
- Training data that could help: More examples of short Python-to-C# translations with style-focused feedback would likely improve idiomatic output.


In [8]:
instruction_results = run_probe_set(
    model=model,
    tokenizer=tokenizer,
    probes=TASK2_PROBES["instruction_following"],
    system_prompt=TASK2_GENERAL_SYSTEM_PROMPT,
    max_new_tokens=500,
)

for result in instruction_results:
    print("=" * 100)
    print(f"ID: {result['id']}")
    print("PROMPT:")
    print(result["prompt"])
    print()
    print("MODEL OUTPUT:")
    print(result["output"])
    print()

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ID: instruction_1
PROMPT:
Answer with exactly 3 bullet points. Each bullet must be at most 6 words. Topic: why version control is useful.

MODEL OUTPUT:
- Tracks changes over time.
- Facilitates collaboration among team members.
- Ensures project integrity through history preservation.

ID: instruction_2
PROMPT:
Return only valid JSON with keys `title` and `year`. Set `title` to `Dune` and `year` to 1965.

MODEL OUTPUT:
```json
{
  "title": "Dune",
  "year": 1965
}
```

ID: instruction_3
PROMPT:
Write one sentence about the ocean. Do not use the letter e.

MODEL OUTPUT:
The vast blue expanse of the ocean stretches endlessly, its depths teeming with mystery and wonder.



### Instruction Following Observations

#### `instruction_1`
- Did it follow the exact format? Yes. It returned exactly 3 bullet points, and each bullet stayed within the 6-word limit.
- Diagnosis: This shows the model can follow simple counting and formatting constraints when they are short and concrete.

#### `instruction_2`
- Did it return only valid JSON? No. The JSON content is correct, but the model wrapped it in a fenced code block, so it did not follow the `only valid JSON` instruction exactly.
- Diagnosis: This is a common instruction-following problem where the model knows the right content but still adds presentation formatting learned from chat-style examples.

#### `instruction_3`
- Did it follow the letter constraint? No. The sentence contains many uses of the letter `e`.
- Diagnosis: The model ignored a fine-grained lexical constraint even though the broader topic instruction was easy. This suggests exact character-level control is weaker than general content generation.


In [9]:
competence_results = run_probe_set(
    model=model,
    tokenizer=tokenizer,
    probes=TASK2_PROBES["surprising_competence"],
    system_prompt=TASK2_GENERAL_SYSTEM_PROMPT,
    max_new_tokens=500,
)

for result in competence_results:
    print("=" * 100)
    print(f"ID: {result['id']}")
    print("PROMPT:")
    print(result["prompt"])
    print()
    print("MODEL OUTPUT:")
    print(result["output"])
    print()

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ID: competence_1
PROMPT:
Explain in simple terms what a SQL LEFT JOIN does, and include one tiny example.

MODEL OUTPUT:
A SQL LEFT JOIN is like combining two tables to see all records from both tables, but it also includes any records that don't have matching values in either table. It's kind of like when you're playing hide-and-seek, and you look for your friend who might be hiding behind different parts of the room - if they're not there, you still check those spots because you want to make sure you haven't missed anyone.

Here's an example:

Let's say we have two tables: "employees" and "departments". We want to find out which departments each employee belongs to. The left join will give us every department (even if some employees don't belong to that department), and then we can add more information about each department by joining with another table that has details on each department.

ID: competence_2
PROMPT:
Write a regular expression that matches a date in the format YYYY-MM-

### Surprising Competence Observations

#### `competence_1`
- Why this output is stronger than expected: The answer is fluent and tries to use a simple analogy, so it sounds accessible at first read.
- Likely reason for this competence: The model seems to have seen many beginner-style database explanations, but the technical content is still shaky because it incorrectly describes a LEFT JOIN as including all records from both tables.

#### `competence_2`
- Why this output is stronger than expected: This is the strongest example in the section because the regex itself is mostly good and the explanation is structured clearly.
- Likely reason for this competence: Regex patterns and date-format explanations are common in public coding data, so the model likely learned this pattern well even though its final note about invalid calendar dates is too strong.

#### `competence_3`
- Why this output is stronger than expected: The model recalls the basic LIFO/FIFO distinction immediately, which shows some surface knowledge of the topic.
- Likely reason for this competence: The model probably memorized common textbook definitions, but the rest of the answer drifts into incorrect claims about stack and queue behavior, so the competence here is only partial.


### Task 2.1 Interpretation

- Main math failure categories seen here: formatting is the clearest one in these four custom probes.
- Which category seems most common? In these particular examples, formatting is the only clear failure because three of the four problems were solved correctly.
- Which examples show reasoning failure? None of the four custom math probes clearly show a reasoning failure.
- Which examples show arithmetic failure? None of the four custom math probes clearly show an arithmetic failure.
- Which examples show formatting failure? `math_3`.
- What kind of GSM8K fine-tuning would likely help most? Training data that combines chain-of-thought reasoning with a strict `#### final_answer` ending would likely help the most.
- What this tells us before fine-tuning: These hand-written probes may be too easy to expose the model's deeper math weaknesses. The model looks solid on short arithmetic word problems, but it already shows that correct reasoning does not guarantee the right output format. The harder GSM8K baseline below is therefore important for finding the real failure pattern.


## Task 2.2 - GSM8K Zero-Shot Baseline

This section evaluates the same base model on 50 shuffled GSM8K test examples using deterministic generation and exact-match answer extraction.

In [10]:
gsm8k = load_dataset("openai/gsm8k", "main")
test_sample = gsm8k["test"].shuffle(seed=SEED).select(range(50))

print({
    "sample_size": len(test_sample),
    "first_question": test_sample[0]["question"],
})

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

{'sample_size': 50, 'first_question': "Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162, calculate Allen's age 10 years from now."}


In [11]:
zero_shot_predictions = []

for index, example in enumerate(test_sample):
    prediction = generate_response(
        model=model,
        tokenizer=tokenizer,
        user_prompt=example["question"],
        system_prompt=GSM8K_ZERO_SHOT_SYSTEM_PROMPT,
        max_new_tokens=500,
        do_sample=False,
    )
    zero_shot_predictions.append(prediction)
    if index < 3:
        print("=" * 100)
        print(f"Example {index}")
        print("QUESTION:")
        print(example["question"])
        print()
        print("MODEL OUTPUT:")
        print(prediction)
        print()
        print("REFERENCE ANSWER:")
        print(example["answer"])
        print()

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Example 0
QUESTION:
Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162, calculate Allen's age 10 years from now.

MODEL OUTPUT:
To solve this problem, we need to follow these steps:

1. **Determine the individual ages based on the given ratio**:
   - The ratio of Darrell's age to Allen's age is 7:11.
   - Let \( D \) represent Darrell's age and \( A \) represent Allen's age.

2. **Set up an equation using the ratio**:
   \[
   \frac{D}{A} = \frac{7}{11}
   \]

3. **Express one variable in terms of the other**:
   \[
   D = \frac{7}{11}A
   \]

4. **Use the total age equation**:
   \[
   D + A = 162
   \]
   Substitute \( D \) from the previous equation into the total age equation:
   \[
   \frac{7}{11}A + A = 162
   \]

5. **Combine like terms**:
   \[
   \left(\frac{7}{11} + 1\right)A = 162
   \]
   Simplify inside the parentheses:
   \[
   \left(\frac{7}{11} + \frac{11}{11}\right)A = 162
   \]
   \[
   \frac{18}{11}A = 162
   \]

6. **Solve for \( A \)**

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Example 1
QUESTION:
Lorraine and Colleen are trading stickers for buttons. Each large sticker is worth a large button or three small buttons. A small sticker is worth one small button. A large button is worth three small stickers. Lorraine starts with 30 small stickers and 40 large stickers. She trades 90% of her small stickers for large buttons. She trades 50% of her large stickers for large buttons and trades the rest of them for small buttons. How many buttons does she have by the end?

MODEL OUTPUT:
To solve this problem, we need to follow these steps:

1. **Calculate the number of small stickers traded:**
   - Lorraine starts with 30 small stickers.
   - She trades 90% of her small stickers.
   \[
   \text{Number of small stickers traded} = 30 \times 0.90 = 27
   \]

2. **Determine the value of the traded small stickers in terms of large buttons:**
   - Each large button is worth 3 small stickers.
   - Therefore, each large button is worth:
     \[
     \frac{3 \text{ small sticke

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Example 2
QUESTION:
Indras has 6 letters in her name. Her sister's name has 4 more letters than half of the letters in Indras' name. How many letters are in Indras and her sister's names?

MODEL OUTPUT:
To find out how many letters are in both Indras and her sister's names, we need to follow these steps:

1. **Calculate the number of letters in Indras' name:**
   - Indras has 6 letters.

2. **Determine the number of letters in her sister's name:**
   - Her sister's name has 4 more letters than half of the letters in Indras' name.
   
3. **Find half of the letters in Indras' name:**
   \[
   \frac{6}{2} = 3
   \]

4. **Add 4 to this value to get the total number of letters in her sister's name:**
   \[
   3 + 4 = 7
   \]

5. **Sum up the total number of letters in both names:**
   \[
   6 (Indras) + 7 (her sister) = 13
   \]

Therefore, the total number of letters in both Indras and her sister's names is \(13\).

Final Answer:
\[
\boxed{13}
\]

REFERENCE ANSWER:
I = <<6=6>>6
Sister = 6/

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

In [12]:
baseline_metrics = evaluate_gsm8k_sample(
    predictions=zero_shot_predictions,
    references=list(test_sample["answer"]),
)

baseline_summary = {
    "accuracy": baseline_metrics["accuracy"],
    "parse_failure_rate": baseline_metrics["parse_failure_rate"],
    "correct": baseline_metrics["correct"],
    "total": baseline_metrics["total"],
}

baseline_summary

{'accuracy': 0.5, 'parse_failure_rate': 0.0, 'correct': 25, 'total': 50}

In [14]:
baseline_rows = pd.DataFrame(baseline_metrics["rows"])
baseline_rows[["predicted_answer", "reference_answer", "correct", "parse_failed"]].head(30)

,predicted_answer,reference_answer,correct,parse_failed
0,108,109,False,False
1,6,89,False,False
2,13,13,True,False
3,5,5,True,False
4,45,25,False,False
5,217,452,False,False
6,43,43,True,False
7,34,34,True,False
8,40,120,False,False
9,66,11,False,False


In [16]:
baseline_rows.loc[
    baseline_rows["correct"] == False,
    ["predicted_answer", "reference_answer", "correct", "parse_failed"]
].head(30)

,predicted_answer,reference_answer,correct,parse_failed
0,108,109,False,False
1,6,89,False,False
4,45,25,False,False
5,217,452,False,False
8,40,120,False,False
9,66,11,False,False
10,42,34,False,False
12,1,15,False,False
13,4,24,False,False
14,70,105,False,False


### Baseline Interpretation

- Baseline 1 zero-shot accuracy: `0.50` exact-match accuracy, which is `25/50`.
- Parse-failure rate: `0.00`, because every sampled output produced a parseable final number.
- How does this compare to the qualitative math behavior from Task 2.1? The custom probes made the model look stronger than it really is because they were short and direct. On the GSM8K sample, the model still produces parseable answers almost every time, but many of those answers are wrong, so the bigger issue is math quality rather than answer extraction.
- Main failure mode to carry into Task 6: The main failure mode is incorrect reasoning or arithmetic on harder multi-step problems, not formatting failure.
- Baseline 1 note for `experiment_log.md`: Zero-shot baseline on 50 shuffled GSM8K test examples reached 50% exact-match accuracy with a 0% parse-failure rate. The model usually returns a usable final number, but many answers are still mathematically wrong on harder reasoning tasks.
